# PlumeTrace PINN Source Attribution - Kaggle Dual T4 Notebook (v2)

Use this notebook on Kaggle with **Accelerator: GPU T4 x2**. It trains the PlumeTrace Physics-Informed Neural Network (PINN) for inverse pollutant-source attribution, saves a model checkpoint, writes a dense source-probability map, and validates whether the predicted source is close to the injected synthetic factory source.

The notebook keeps the original 2D advection-diffusion physics loss:

`dC/dt + u*dC/dx + v*dC/dy - D*(d2C/dx2 + d2C/dy2) = 0`

For numerical stability, training runs in float32 instead of mixed precision, because second-order PINN derivatives can become noisy in fp16 - this was true in v1 and remains true here; adding `torch.cuda.amp` would trade away exactly the stability this architecture needs.

### What's new in this refactor
- **Architecture factory** (`ModelConfig.arch_type`): `fourier_residual` (default - Fourier features + residual MLP + learnable-steepness Swish), `residual`, `siren`, or `mlp` (original baseline), switchable with one config field.
- **SoftAdapt adaptive loss balancing**: data/physics/boundary weights are re-derived every epoch from their relative rate of improvement instead of being hand-tuned once.
- **Residual-based Adaptive Refinement (RAR)**: periodically resamples collocation points toward the highest-PDE-residual regions of the domain.
- **Sobol quasi-random sampling** for collocation/boundary points (lower discrepancy than uniform random for the same point budget).
- **Two-stage optimization**: AdamW + warmup/cosine schedule, then a full-batch L-BFGS fine-tune pass.
- **EMA of weights**: the checkpointed/evaluated model is an exponential moving average, not the raw last-step weights.
- **Local MLflow tracking** (SQLite-backed, no account needed) alongside the existing CSV/JSON/plot outputs.
- New training-diagnostics figure (loss components, adaptive weights, LR schedule, gradient norm) next to the original probability-map figure.

### Explicitly out of scope for this pass
Distributed Optuna sweeps, W&B, MLflow-remote, Bayesian PINN / deep ensembles / K-fold cross-validation, a full ablation matrix across every architecture pair, CUDA graphs, and `torch.compile` were left out on purpose - they either need infrastructure a single Kaggle session doesn't have (accounts, multi-run orchestration) or would multiply the runtime far beyond what "keep it fully runnable end-to-end" allows. The architecture factory and config make it straightforward to add an outer sweep loop later if you want it.

In [ ]:
import os
os.makedirs('pinn', exist_ok=True)
with open('pinn/__init__.py', 'w') as f:
    pass
print('pinn package directory initialized.')

In [ ]:
%%writefile pinn/config.py
"""
PlumeTrace PINN - Configuration System
Research-grade configuration using dataclasses with full type hints.
All hyperparameters are centralized here for reproducibility.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal, List, Tuple


@dataclass(frozen=True)
class SensorStation:
    """Deterministic virtual sensor location inside the industrial sector."""
    sensor_id: str
    latitude: float
    longitude: float


@dataclass(frozen=True)
class CitySector:
    """Geographic bounds and known demo source for the synthetic scenario."""
    lat_min: float = 40.7040
    lat_max: float = 40.7220
    lon_min: float = -74.0160
    lon_max: float = -73.9940
    source_latitude: float = 40.7138
    source_longitude: float = -74.0072


@dataclass(frozen=True)
class ModelConfig:
    """Neural network architecture configuration."""
    arch_type: Literal[
        'mlp', 'residual', 'fourier_residual', 'siren'
    ] = 'fourier_residual'
    hidden_units: int = 128
    hidden_layers: int = 8          # Match notebook 8 layers
    fourier_bands: int = 8          # Match notebook 8 frequency bands
    fourier_sigma: float = 4.0      # Match notebook sigma=4.0
    siren_omega0_first: float = 30.0
    siren_omega0_hidden: float = 30.0
    use_adaptive_activation: bool = True
    use_layer_norm: bool = True
    use_attention: bool = False     # Optional attention block after residual layers
    dropout: float = 0.0
    # Gradient checkpointing gated by model_config flag
    use_grad_checkpoint: bool = False


@dataclass(frozen=True)
class TrainingConfig:
    """Optimization and physics settings for the inversion experiment."""
    # ── Epochs & Stages ─────────────────────────────────────────────
    epochs: int = 2500              # Total AdamW epochs
    learning_rate: float = 1.0e-3
    weight_decay: float = 1.0e-5

    # ── Stage-2 L-BFGS fine-tuning ──────────────────────────────────
    lbfgs_steps: int = 200          # Maximum iterations for one-shot L-BFGS call
    lbfgs_lr: float = 0.5

    # ── Learning Rate ────────────────────────────────────────────────
    warmup_epochs: int = 100
    min_lr_ratio: float = 0.08      # Cosine annealing minimum ratio (eta_min ratio)
    use_reduce_lr_on_plateau: bool = False
    reduce_lr_patience: int = 150
    reduce_lr_factor: float = 0.5

    # ── Loss Weights (Base; adaptive methods override dynamically) ──
    lambda_data: float = 1.0
    lambda_physics: float = 0.10
    lambda_boundary: float = 0.02
    lambda_initial: float = 0.0   # Initial condition loss weight
    lambda_source: float = 0.0      # Source localization prior loss

    # ── Adaptive Weighting ────────────────────────────────────────────
    adapt_every: int = 1
    adapt_method: Literal['softadapt', 'gradnorm', 'ntk', 'fixed'] = 'softadapt'
    softadapt_beta: float = 0.1
    softadapt_floor: float = 0.05
    softadapt_ceil: float = 5.0
    softadapt_warmup_epochs: int = 20
    softadapt_window_size: int = 20  # Number of historical epochs for rate computation
    gradnorm_alpha: float = 1.5
    gradient_clip_norm: float = 5.0

    # ── Sampling Points ─────────────────────────────────────────────
    collocation_points: int = 2000  # Default collocation count
    boundary_points: int = 1024
    initial_points: int = 512     # Points for t=0 initial condition
    source_prior_points: int = 256  # Points for source localization prior

    # ── RAR (Residual Adaptive Refinement) ──────────────────────────
    rar_interval: int = 100         # Refresh high-residual points every N epochs (rar_every)
    rar_eval_points: int = 50000
    rar_add_points: int = 1000
    rar_max_points: int = 32000
    rar_fraction: float = 0.25
    rar_pool_multiplier: int = 4
    rar_warmup_epochs: int = 200

    # ── Logging & Checkpointing ───────────────────────────────────
    log_every: int = 50
    checkpoint_every: int = 500
    early_stop_patience: int = 300
    ema_decay: float = 0.999
    tensorboard_dir: str = './tensorboard'
    output_dir: str = './outputs'

    # ── Grid & Evaluation ─────────────────────────────────────────
    grid_size: int = 220
    eval_time_samples: List[float] = field(default_factory=lambda: [0.0, 0.05, 0.1])
    probability_temperature: float = 0.010

    # ── Physics Base ────────────────────────────────────────────────
    wind_u: float = 0.16
    wind_v: float = -0.06
    diffusion: float = 0.006
    sensor_time_samples: int = 96
    random_seed: int = 2026

    # ── Data Augmentation ───────────────────────────────────────────
    # NEW: Randomized physics parameters for curriculum learning
    augment_wind_std: float = 0.03
    augment_diffusion_std: float = 0.001
    augment_diffusion_min: float = 0.004
    augment_diffusion_max: float = 0.010
    sensor_noise_std: float = 1.65
    background_so2_ppb: float = 7.5
    event_scale_ppb: float = 180.0
    missing_sensor_prob: float = 0.0   # Probability of dropping a sensor reading
    temporal_jitter_std: float = 0.0     # Std of time perturbation

    # ── Boundary Conditions ─────────────────────────────────────────
    bc_type: Literal['dirichlet', 'neumann', 'robin', 'mixed'] = 'dirichlet'
    bc_dirichlet_value: float = 0.0
    bc_robin_alpha: float = 1.0
    bc_robin_beta: float = 0.0
    hard_bc_enforcement: bool = False    # Transform output to satisfy BCs exactly

    # ── Multi-GPU ────────────────────────────────────────────────────
    use_data_parallel: bool = False
    use_distributed: bool = False
    local_rank: int = 0

    # ── Mixed Precision ─────────────────────────────────────────────
    use_amp: bool = True
    amp_dtype: Literal['float16', 'bfloat16'] = 'bfloat16'

    # ── Reproducibility ─────────────────────────────────────────────
    deterministic: bool = True
    cudnn_benchmark: bool = False   # FIX: Disabled to ensure reproducibility

    # ── Curriculum Learning ─────────────────────────────────────────
    curriculum_epochs: int = 500    # Epochs before full complexity
    curriculum_noise_max: float = 0.5  # Noise multiplier during curriculum

    # ── Ensemble / Uncertainty ────────────────────────────────────────
    mc_dropout_samples: int = 0     # 0 = disabled
    ensemble_size: int = 1          # 1 = disabled

    # ── Validation ────────────────────────────────────────────────────
    val_split: float = 0.2


# ── Default Sensor Stations ─────────────────────────────────────────
SENSOR_STATIONS: Tuple[SensorStation, ...] = (
    SensorStation('industrial_north', 40.7180, -74.0060),
    SensorStation('residential_east', 40.7140, -73.9980),
    SensorStation('park_south', 40.7080, -74.0040),
    SensorStation('river_west', 40.7120, -74.0120),
)

SECTOR = CitySector()


In [ ]:
%%writefile pinn/utils.py
"""
PlumeTrace PINN - Utilities
Reproducibility, EMA, checkpointing, logging, metrics, and helper functions.
"""
from __future__ import annotations

from collections import deque
import json
import logging
import math
import os
import random
import time
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import torch
from torch import Tensor, nn

LOGGER = logging.getLogger("plumetrace.pinn")


# ── Reproducibility ─────────────────────────────────────────────────

def configure_logging(level: int = logging.INFO) -> None:
    """Configure root logger with a consistent format."""
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )


def seed_everything(seed: int, deterministic: bool = True) -> None:
    """
    Set all random seeds for reproducibility.
    FIX: Disable cudnn.benchmark to ensure deterministic behavior.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False  # FIX: was True, caused non-determinism
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')


def get_device(local_rank: int = 0) -> torch.device:
    """Select the best available device."""
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{local_rank}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    LOGGER.info("Using device: %s", device)
    return device


# ── Model Helpers ─────────────────────────────────────────────────

def unwrap_model(model: nn.Module) -> nn.Module:
    """Unwrap DataParallel / DistributedDataParallel / Compiled models."""
    if isinstance(model, (nn.DataParallel, nn.parallel.DistributedDataParallel)):
        model = model.module
    if hasattr(model, '_orig_mod'):
        model = model._orig_mod
    return model


def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── EMA (Exponential Moving Average) ──────────────────────────────

class EMA:
    """
    Exponential Moving Average of model parameters and buffers.
    Maintains a shadow copy of state dict that updates slowly.
    """
    def __init__(self, model: nn.Module, decay: float = 0.999) -> None:
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in unwrap_model(model).state_dict().items()}

    def update(self, model: nn.Module) -> None:
        """Update shadow state dict with EMA formula."""
        for k, v in unwrap_model(model).state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module) -> None:
        """Copy EMA state dict into the model."""
        unwrap_model(model).load_state_dict(self.shadow, strict=True)

    def state_dict(self) -> Dict[str, Tensor]:
        return {k: v.clone() for k, v in self.shadow.items()}

    def load_state_dict(self, state_dict: Dict[str, Tensor]) -> None:
        self.shadow = {k: v.clone() for k, v in state_dict.items()}


# ── SoftAdapt Dynamic Weighting ───────────────────────────────────

class SoftAdaptWeighter:
    """
    SoftAdapt dynamic loss weighting (Heydari et al., 2019).
    Reweights loss components based on their rate of change over a rolling window.
    """
    def __init__(self, names: list[str], beta: float, floor: float, ceil: float, window_size: int = 20) -> None:
        self.names = names
        self.beta = beta
        self.floor = floor
        self.ceil = ceil
        self.window_size = max(2, window_size)
        self.history: deque[dict[str, float]] = deque(maxlen=self.window_size)

    def compute(self, current: dict[str, float]) -> dict[str, float]:
        self.history.append(dict(current))
        if len(self.history) < 2:
            return {n: 1.0 for n in self.names}
        
        first = self.history[0]
        last = self.history[-1]
        
        rates = []
        for n in self.names:
            first_val = first[n]
            last_val = last[n]
            rate = (last_val - first_val) / (abs(first_val) + 1e-8)
            rates.append(rate)
            
        rates = np.array(rates)
        shifted = rates - rates.max()
        exp = np.exp(self.beta * shifted)
        raw = np.clip(exp / exp.sum() * len(self.names), self.floor, self.ceil)
        return dict(zip(self.names, raw.tolist()))


# ── Checkpointing ─────────────────────────────────────────────────

class CheckpointManager:
    """Handles saving and loading model checkpoints with automatic resume."""

    def __init__(self, output_dir: Path, keep_best: bool = True) -> None:
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.keep_best = keep_best
        self.best_score = float('inf')
        self.checkpoint_path = self.output_dir / "latest_checkpoint.pt"
        self.best_path = self.output_dir / "best_model.pt"

    def save(
        self,
        model: nn.Module,
        optimizer: Optional[Any],
        scheduler: Optional[Any],
        ema: Optional[EMA],
        epoch: int,
        score: float,
        config: Optional[Any] = None,
    ) -> None:
        """Save a checkpoint with all training state."""
        state = {
            'epoch': epoch,
            'model_state_dict': unwrap_model(model).state_dict(),
            'score': score,
        }
        if optimizer is not None:
            state['optimizer_state_dict'] = optimizer.state_dict()
        if scheduler is not None:
            state['scheduler_state_dict'] = scheduler.state_dict()
        if ema is not None:
            state['ema_state_dict'] = ema.state_dict()
        if config is not None:
            from dataclasses import asdict
            state['config'] = asdict(config)

        torch.save(state, self.checkpoint_path)
        LOGGER.info("Checkpoint saved at epoch %d (score: %.6f)", epoch, score)

        if self.keep_best and score < self.best_score:
            self.best_score = score
            torch.save(state, self.best_path)
            LOGGER.info("New best model saved (score: %.6f)", score)

    def load(self, model: nn.Module, device: torch.device) -> Optional[Dict[str, Any]]:
        """Load checkpoint and return state dict if found."""
        if self.checkpoint_path.exists():
            LOGGER.info("Resuming from checkpoint: %s", self.checkpoint_path)
            checkpoint = torch.load(self.checkpoint_path, map_location=device, weights_only=False)
            unwrap_model(model).load_state_dict(checkpoint['model_state_dict'])
            return checkpoint
        return None

    def load_best(self, model: nn.Module, device: torch.device) -> bool:
        """Load best model if it exists."""
        if self.best_path.exists():
            checkpoint = torch.load(self.best_path, map_location=device, weights_only=False)
            unwrap_model(model).load_state_dict(checkpoint['model_state_dict'])
            LOGGER.info("Loaded best model from %s", self.best_path)
            return True
        return False


# ── Metrics ────────────────────────────────────────────────────────

def compute_r2(pred: np.ndarray, true: np.ndarray) -> float:
    """Compute R² coefficient of determination."""
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - np.mean(true)) ** 2)
    return float(1.0 - ss_res / (ss_tot + 1e-8))


def compute_rmse(pred: np.ndarray, true: np.ndarray) -> float:
    """Compute Root Mean Square Error."""
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def compute_mae(pred: np.ndarray, true: np.ndarray) -> float:
    """Compute Mean Absolute Error."""
    return float(np.mean(np.abs(pred - true)))


def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Compute great-circle distance between two lat/lon points in meters."""
    radius = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    return 2.0 * radius * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))


def normalize_scores(scores: np.ndarray) -> np.ndarray:
    """Normalize scores to a valid probability distribution."""
    clipped = np.clip(scores.astype(np.float64), 0.0, None)
    total = float(clipped.sum())
    if not np.isfinite(total) or total <= 0.0:
        return np.full_like(clipped, 1.0 / clipped.size, dtype=np.float64)
    return clipped / total


# ── Coordinate Transforms ──────────────────────────────────────────

def lat_lon_to_normalized(
    latitude: float | np.ndarray,
    longitude: float | np.ndarray,
    lat_min: float, lat_max: float, lon_min: float, lon_max: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Convert latitude/longitude to normalized [0,1] coordinates."""
    lon_arr = np.asarray(longitude, dtype=np.float32)
    lat_arr = np.asarray(latitude, dtype=np.float32)
    x = (lon_arr - lon_min) / (lon_max - lon_min)
    y = (lat_arr - lat_min) / (lat_max - lat_min)
    return x.astype(np.float32), y.astype(np.float32)


def normalized_to_lat_lon(
    x: np.ndarray,
    y: np.ndarray,
    lat_min: float, lat_max: float, lon_min: float, lon_max: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Convert normalized [0,1] coordinates back to latitude/longitude."""
    longitude = lon_min + x * (lon_max - lon_min)
    latitude = lat_min + y * (lat_max - lat_min)
    return latitude.astype(np.float32), longitude.astype(np.float32)


# ── Timer Context Manager ─────────────────────────────────────────

class Timer:
    """Simple context manager for timing code blocks."""
    def __init__(self, name: str = "Operation") -> None:
        self.name = name
        self.start: Optional[float] = None

    def __enter__(self) -> Timer:
        self.start = time.time()
        return self

    def __exit__(self, *args: Any) -> None:
        elapsed = time.time() - self.start
        LOGGER.info("%s completed in %.3f seconds", self.name, elapsed)


# ── JSON Logger for Experiment Tracking ───────────────────────────

class ExperimentLogger:
    """Log hyperparameters and results to JSON for reproducibility."""
    def __init__(self, output_dir: Path) -> None:
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.log_path = self.output_dir / "experiment_log.json"
        self.records: list[Dict[str, Any]] = []

    def log(self, record: Dict[str, Any]) -> None:
        self.records.append(record)
        self.log_path.write_text(json.dumps(self.records, indent=2), encoding='utf-8')


In [ ]:
%%writefile pinn_engine.py
"""
PlumeTrace physics-informed neural network engine.

This script is a self-contained hackathon demo for inverse-dispersion source
attribution. It trains a Physics-Informed Neural Network (PINN) to reconstruct
the origin of a pollutant plume using sparse measurements from four municipal
IoT sensors and a physics loss derived from the 2D advection-diffusion PDE.

Mathematical model
------------------
The pollutant concentration C(x, y, t) is governed by:

    dC/dt + u * dC/dx + v * dC/dy - D * (d2C/dx2 + d2C/dy2) = 0

The neural network approximates C(x, y, t). During training, the data loss fits
sparse sensor readings while the physics loss penalizes PDE residual error over
Sobol collocation points. The architecture uses advanced Fourier Residual blocks
and two-stage AdamW + L-BFGS optimization.
"""

from __future__ import annotations

import argparse
import logging
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import numpy as np

try:
    import torch
    from torch import Tensor, nn
    from torch.nn import functional as F
    from torch.quasirandom import SobolEngine
except ImportError as exc:
    raise SystemExit(
        "PyTorch is required to run pinn_engine.py. Install it with the "
        "appropriate command from https://pytorch.org/get-started/locally/."
    ) from exc


LOGGER = logging.getLogger("plumetrace.pinn")


from pinn.config import SensorStation, CitySector, ModelConfig, TrainingConfig, SENSOR_STATIONS, SECTOR
from pinn.utils import (
    seed_everything as set_reproducibility,
    get_device,
    configure_logging,
    lat_lon_to_normalized as _lat_lon_to_normalized,
    normalized_to_lat_lon as _normalized_to_lat_lon,
    EMA,
    SoftAdaptWeighter
)


@dataclass
class SyntheticSensorDataset:
    """Tensor and metadata bundle for sparse sensor observations."""
    features: Tensor
    concentration: Tensor
    concentration_scale_ppb: float
    rows: list[dict[str, Any]]
    val_features: Tensor | None = None
    val_concentration: Tensor | None = None


@dataclass
class SourceProbabilityMap:
    """Dense latitude, longitude, and normalized source probability matrices."""
    latitudes: np.ndarray
    longitudes: np.ndarray
    probabilities: np.ndarray


def lat_lon_to_normalized(
    latitude: float | np.ndarray,
    longitude: float | np.ndarray,
    sector: CitySector,
) -> tuple[np.ndarray, np.ndarray]:
    return _lat_lon_to_normalized(latitude, longitude, sector.lat_min, sector.lat_max, sector.lon_min, sector.lon_max)


def normalized_to_lat_lon(
    x: np.ndarray,
    y: np.ndarray,
    sector: CitySector,
) -> tuple[np.ndarray, np.ndarray]:
    return _normalized_to_lat_lon(x, y, sector.lat_min, sector.lat_max, sector.lon_min, sector.lon_max)


# ============================================================
# Architecture factory
# ============================================================
class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int, num_bands: int, sigma: float, seed: int = 0) -> None:
        super().__init__()
        generator = torch.Generator().manual_seed(seed)
        b_matrix = torch.randn((in_dim, num_bands), generator=generator) * sigma
        self.register_buffer('b_matrix', b_matrix)

    def forward(self, x: Tensor) -> Tensor:
        projected = 2.0 * math.pi * (x @ self.b_matrix)
        return torch.cat([torch.sin(projected), torch.cos(projected)], dim=-1)


class AdaptiveSwish(nn.Module):
    def __init__(self, num_features: int) -> None:
        super().__init__()
        self.beta = nn.Parameter(torch.ones(num_features))

    def forward(self, x: Tensor) -> Tensor:
        return x * torch.sigmoid(self.beta * x)


class ResidualBlock(nn.Module):
    def __init__(self, width: int, use_adaptive_activation: bool) -> None:
        super().__init__()
        self.linear1 = nn.Linear(width, width)
        self.linear2 = nn.Linear(width, width)
        self.act1 = AdaptiveSwish(width) if use_adaptive_activation else nn.Tanh()
        self.act2 = AdaptiveSwish(width) if use_adaptive_activation else nn.Tanh()
        self.layer_norm1 = nn.LayerNorm(width)
        self.layer_norm2 = nn.LayerNorm(width)

    def forward(self, x: Tensor) -> Tensor:
        h = self.act1(self.linear1(self.layer_norm1(x)))
        h = self.linear2(self.layer_norm2(h))
        return self.act2(h + x)


class SineLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, omega0: float, is_first: bool) -> None:
        super().__init__()
        self.omega0 = omega0
        self.linear = nn.Linear(in_features, out_features)
        with torch.no_grad():
            bound = (1.0 / in_features) if is_first else (math.sqrt(6.0 / in_features) / omega0)
            self.linear.weight.uniform_(-bound, bound)
            nn.init.zeros_(self.linear.bias)

    def forward(self, x: Tensor) -> Tensor:
        return torch.sin(self.omega0 * self.linear(x))


class PlumeInversionPINN(nn.Module):
    def __init__(self, model_config: ModelConfig) -> None:
        super().__init__()
        self.config = model_config
        arch = model_config.arch_type
        in_dim = 3

        if arch == 'fourier_residual':
            self.encoder: nn.Module | None = FourierFeatures(in_dim, model_config.fourier_bands, model_config.fourier_sigma)
            feat_dim = 2 * model_config.fourier_bands
        else:
            self.encoder = None
            feat_dim = in_dim

        if arch == 'siren':
            n_hidden = max(model_config.hidden_layers - 1, 1)
            siren_layers = [SineLayer(in_dim, model_config.hidden_units, model_config.siren_omega0_first, is_first=True)]
            siren_layers += [
                SineLayer(model_config.hidden_units, model_config.hidden_units, model_config.siren_omega0_hidden, is_first=False)
                for _ in range(n_hidden)
            ]
            self.backbone: nn.Module = nn.Sequential(*siren_layers)
            self.blocks = None
            out_width = model_config.hidden_units
        elif arch in ('residual', 'fourier_residual'):
            self.input_proj = nn.Linear(feat_dim, model_config.hidden_units)
            self.input_act = AdaptiveSwish(model_config.hidden_units) if model_config.use_adaptive_activation else nn.Tanh()
            n_blocks = max(model_config.hidden_layers, 1)
            self.blocks = nn.ModuleList(
                [ResidualBlock(model_config.hidden_units, model_config.use_adaptive_activation) for _ in range(n_blocks)]
            )
            out_width = model_config.hidden_units
        else:
            layers: list[nn.Module] = []
            width = feat_dim
            for _ in range(model_config.hidden_layers):
                layers.append(nn.Linear(width, model_config.hidden_units))
                layers.append(AdaptiveSwish(model_config.hidden_units) if model_config.use_adaptive_activation else nn.Tanh())
                width = model_config.hidden_units
            self.backbone = nn.Sequential(*layers)
            self.blocks = None
            out_width = model_config.hidden_units

        self.output_layer = nn.Linear(out_width, 1)
        self._initialize_weights()
        if self.output_layer.bias is not None:
            nn.init.constant_(self.output_layer.bias, -5.0)

    def _initialize_weights(self) -> None:
        skip_ids = set()
        if self.config.arch_type == 'siren':
            for module in self.backbone:
                skip_ids.add(id(module.linear))
        for module in self.modules():
            if isinstance(module, nn.Linear) and id(module) not in skip_ids:
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: Tensor, y: Tensor | None = None, t: Tensor | None = None) -> Tensor:
        if y is None and t is None:
            features = x
        elif y is not None and t is not None:
            features = torch.cat((x, y, t), dim=1)
        else:
            raise ValueError("Provide either a single feature tensor or all x, y, t tensors.")

        if features.ndim != 2 or features.shape[1] != 3:
            raise ValueError("Model input must have shape [batch, 3]")
        arch = self.config.arch_type
        if arch == 'siren':
            hidden = self.backbone(features)
        elif arch in ('residual', 'fourier_residual'):
            encoded = self.encoder(features) if self.encoder is not None else features
            hidden = self.input_act(self.input_proj(encoded))
            for block in self.blocks:
                if getattr(self.config, 'use_grad_checkpoint', False) and self.training:
                    hidden = torch.utils.checkpoint.checkpoint(block, hidden, use_reentrant=False)
                else:
                    hidden = block(hidden)
        else:
            encoded = self.encoder(features) if self.encoder is not None else features
            hidden = self.backbone(encoded)
        return F.softplus(self.output_layer(hidden), beta=1.0)


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


# ============================================================
# Physics residual
# ============================================================
def compute_pde_residual(
    features: Tensor, model: nn.Module, u: float, v: float, diffusion: float, reduction: Literal['mean', 'none'] = 'mean', create_graph: bool = True
) -> Tensor:
    features = features.clone().requires_grad_(True)
    concentration = model(features)
    ones = torch.ones_like(concentration)
    gradients = torch.autograd.grad(concentration, features, grad_outputs=ones, create_graph=create_graph, retain_graph=create_graph)[0]
    dC_dx, dC_dy, dC_dt = gradients[:, 0:1], gradients[:, 1:2], gradients[:, 2:3]
    d2C_dx2 = torch.autograd.grad(dC_dx, features, grad_outputs=torch.ones_like(dC_dx), create_graph=create_graph, retain_graph=create_graph)[0][:, 0:1]
    d2C_dy2 = torch.autograd.grad(dC_dy, features, grad_outputs=torch.ones_like(dC_dy), create_graph=create_graph, retain_graph=create_graph)[0][:, 1:2]
    residual = dC_dt + u * dC_dx + v * dC_dy - diffusion * (d2C_dx2 + d2C_dy2)
    squared = residual.square()
    return squared.mean() if reduction == 'mean' else squared


# ============================================================
# Samplers (Sobol)
# ============================================================
def sample_collocation_points(count: int, device: torch.device, engine: SobolEngine) -> Tensor:
    return engine.draw(count).to(device=device, dtype=torch.float32)

def sample_boundary_points(count: int, device: torch.device, engine: SobolEngine) -> Tensor:
    n = max(count // 4, 1)
    free = engine.draw(n * 2).to(device=device, dtype=torch.float32)
    t, other = free[:n, 0:1], free[n:, 0:1]
    side0 = torch.cat([torch.zeros((n, 1), device=device), other, t], dim=1)
    side1 = torch.cat([torch.ones((n, 1), device=device), other, t], dim=1)
    side2 = torch.cat([other, torch.zeros((n, 1), device=device), t], dim=1)
    side3 = torch.cat([other, torch.ones((n, 1), device=device), t], dim=1)
    return torch.cat([side0, side1, side2, side3], dim=0)

def rar_select_points(model: nn.Module, config: TrainingConfig, device: torch.device, engine: SobolEngine) -> Tensor:
    pool_size = config.collocation_points * config.rar_pool_multiplier
    candidates = engine.draw(pool_size).to(device=device, dtype=torch.float32)
    with torch.enable_grad():
        residuals = compute_pde_residual(candidates, model, config.wind_u, config.wind_v, config.diffusion, reduction='none', create_graph=False).detach()
    top_k = max(int(config.collocation_points * config.rar_fraction), 1)
    top_idx = torch.topk(residuals.squeeze(-1), k=min(top_k, pool_size), largest=True).indices
    return candidates[top_idx].detach()


# SoftAdaptWeighter and EMA are imported from pinn.utils


def build_warmup_cosine_lambda(warmup_epochs: int, total_epochs: int, min_lr_ratio: float):
    def fn(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine
    return fn


def analytic_advection_diffusion_plume(
    x: np.ndarray,
    y: np.ndarray,
    t: np.ndarray,
    source_x: float,
    source_y: float,
    wind_u: float,
    wind_v: float,
    diffusion: float,
    source_strength: float = 1.0,
    initial_spread: float = 0.035,
) -> np.ndarray:
    effective_time = np.maximum(t, 0.0) + initial_spread
    advected_x = source_x + wind_u * t
    advected_y = source_y + wind_v * t
    radial_distance_squared = (x - advected_x) ** 2 + (y - advected_y) ** 2
    denominator = 4.0 * math.pi * diffusion * effective_time
    exponent = -radial_distance_squared / (4.0 * diffusion * effective_time)
    return source_strength * np.exp(exponent) / denominator


def generate_synthetic_sensor_data(
    sector: CitySector,
    config: TrainingConfig,
    device: torch.device,
) -> SyntheticSensorDataset:
    source_x_arr, source_y_arr = lat_lon_to_normalized(sector.source_latitude, sector.source_longitude, sector)
    source_x = float(np.asarray(source_x_arr))
    source_y = float(np.asarray(source_y_arr))

    rows: list[dict[str, Any]] = []
    x_values: list[float] = []
    y_values: list[float] = []
    t_values: list[float] = []
    plume_signal_values: list[float] = []

    sample_times = np.linspace(0.05, 1.0, config.sensor_time_samples, dtype=np.float32)
    for sensor in SENSOR_STATIONS:
        sensor_x, sensor_y = lat_lon_to_normalized(sensor.latitude, sensor.longitude, sector)
        sx = float(np.asarray(sensor_x))
        sy = float(np.asarray(sensor_y))

        for timestamp in sample_times:
            signal = analytic_advection_diffusion_plume(
                np.asarray(sx, dtype=np.float32),
                np.asarray(sy, dtype=np.float32),
                np.asarray(float(timestamp), dtype=np.float32),
                source_x,
                source_y,
                config.wind_u,
                config.wind_v,
                config.diffusion,
            )
            x_values.append(sx)
            y_values.append(sy)
            t_values.append(float(timestamp))
            plume_signal_values.append(float(signal))

    plume = np.asarray(plume_signal_values, dtype=np.float32)
    normalized_signal = plume / max(float(plume.max()), 1.0e-8)
    background_so2_ppb = 7.5
    event_scale_ppb = 180.0
    noise = np.random.normal(loc=0.0, scale=1.65, size=normalized_signal.shape)
    so2_ppb = np.clip(background_so2_ppb + event_scale_ppb * normalized_signal + noise, 0.0, None)
    concentration_scale_ppb = float(np.max(so2_ppb))
    target = (so2_ppb / concentration_scale_ppb).astype(np.float32)

    for i, (x, y, t, so2, c) in enumerate(zip(x_values, y_values, t_values, so2_ppb, target)):
        sensor = SENSOR_STATIONS[i // config.sensor_time_samples]
        rows.append({
            'sensor_id': sensor.sensor_id,
            'latitude': sensor.latitude,
            'longitude': sensor.longitude,
            'x': float(x),
            'y': float(y),
            'elapsed_time': float(t),
            'so2_ppb': float(so2),
            'normalized_concentration': float(c),
        })

    features = torch.tensor(np.column_stack([x_values, y_values, t_values]), dtype=torch.float32, device=device)
    concentration = torch.tensor(target, dtype=torch.float32, device=device).view(-1, 1)

    indices = torch.randperm(len(features))
    split_idx = int(len(features) * 0.8)
    train_idx, val_idx = indices[:split_idx], indices[split_idx:]
    
    val_features = features[val_idx]
    val_concentration = concentration[val_idx]
    features = features[train_idx]
    concentration = concentration[train_idx]

    LOGGER.info("Generated %d train, %d val synthetic readings from %d sensors", len(train_idx), len(val_idx), len(SENSOR_STATIONS))
    return SyntheticSensorDataset(features, concentration, concentration_scale_ppb, rows, val_features, val_concentration)


def train_inversion_model(
    model: nn.Module,
    dataset: SyntheticSensorDataset,
    config: TrainingConfig,
    device: torch.device,
    epochs_override: int | None = None,
) -> tuple[nn.Module, list[dict[str, float]]]:
    epochs = epochs_override if epochs_override is not None else config.epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, build_warmup_cosine_lambda(config.warmup_epochs, epochs, config.min_lr_ratio))
    weighter = SoftAdaptWeighter(
        ['data', 'physics', 'boundary'],
        config.softadapt_beta,
        config.softadapt_floor,
        config.softadapt_ceil,
        window_size=getattr(config, 'softadapt_window_size', 20)
    )
    ema = EMA(model, config.ema_decay)

    colloc_engine = SobolEngine(dimension=3, scramble=True, seed=config.random_seed)
    bound_engine = SobolEngine(dimension=2, scramble=True, seed=config.random_seed + 1)
    rar_engine = SobolEngine(dimension=3, scramble=True, seed=config.random_seed + 2)

    base_lambdas = {'data': config.lambda_data, 'physics': config.lambda_physics, 'boundary': config.lambda_boundary}
    rar_points: Tensor | None = None
    history: list[dict[str, float]] = []
    adaptive_weights = {'data': 1.0, 'physics': 1.0, 'boundary': 1.0}

    LOGGER.info("Stage 1/2 (AdamW): %d epochs", epochs)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        predicted = model(dataset.features)
        data_loss = F.mse_loss(predicted, dataset.concentration)

        n_random = config.collocation_points if rar_points is None else max(config.collocation_points - rar_points.shape[0], 0)
        random_colloc = sample_collocation_points(n_random, device, colloc_engine)
        collocation = random_colloc if rar_points is None else torch.cat([random_colloc, rar_points], dim=0)
        physics_loss = compute_pde_residual(collocation, model, config.wind_u, config.wind_v, config.diffusion)

        boundary = sample_boundary_points(config.boundary_points, device, bound_engine)
        boundary_loss = model(boundary).square().mean()

        current_losses = {
            'data': float(data_loss.detach().cpu()),
            'physics': float(physics_loss.detach().cpu()),
            'boundary': float(boundary_loss.detach().cpu()),
        }
        if epoch > config.softadapt_warmup_epochs:
            adaptive_weights = weighter.compute(current_losses)
        else:
            weighter.prev = dict(current_losses)
            adaptive_weights = {'data': 1.0, 'physics': 1.0, 'boundary': 1.0}

        total_loss = (
            base_lambdas['data'] * adaptive_weights['data'] * data_loss
            + base_lambdas['physics'] * adaptive_weights['physics'] * physics_loss
            + base_lambdas['boundary'] * adaptive_weights['boundary'] * boundary_loss
        )
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
        optimizer.step()
        ema.update(model)
        scheduler.step()

        # Validation tracking & Early Stopping
        val_loss_val = 0.0
        if dataset.val_features is not None:
            with torch.no_grad():
                val_pred = model(dataset.val_features)
                val_loss_val = float(F.mse_loss(val_pred, dataset.val_concentration).cpu())

        history.append(
            {
                'epoch': float(epoch),
                'total_loss': float(total_loss.detach().cpu()),
                'val_loss': val_loss_val,
                'data_loss': current_losses['data'],
                'physics_loss': current_losses['physics'],
                'boundary_loss': current_losses['boundary'],
                'weight_data': float(base_lambdas['data'] * adaptive_weights['data']),
                'weight_physics': float(base_lambdas['physics'] * adaptive_weights['physics']),
                'weight_boundary': float(base_lambdas['boundary'] * adaptive_weights['boundary']),
                'learning_rate': float(scheduler.get_last_lr()[0]),
                'grad_norm': float(grad_norm.detach().cpu() if isinstance(grad_norm, Tensor) else grad_norm),
            }
        )

        if epoch > config.rar_warmup_epochs and epoch % config.rar_interval == 0:
            rar_points = rar_select_points(model, config, device, rar_engine)

        if epoch == 1 or epoch % config.log_every == 0 or epoch == epochs:
            LOGGER.info(
                "Epoch %04d | total_loss=%.6f | val=%.6f | data=%.6f | physics=%.6f | boundary=%.6f",
                epoch,
                float(total_loss.detach().cpu()),
                val_loss_val,
                current_losses['data'],
                current_losses['physics'],
                current_losses['boundary'],
            )

    LOGGER.info("Stage 2/2 (L-BFGS): %d steps", config.lbfgs_steps)
    if not history:
        history.append(
            {
                'epoch': 0.0,
                'total_loss': math.nan,
                'data_loss': math.nan,
                'physics_loss': math.nan,
                'boundary_loss': math.nan,
                'weight_data': 1.0,
                'weight_physics': 1.0,
                'weight_boundary': 1.0,
                'learning_rate': 0.0,
                'grad_norm': math.nan,
            }
        )

    final_weights = {
        'data': base_lambdas['data'] * adaptive_weights['data'],
        'physics': base_lambdas['physics'] * adaptive_weights['physics'],
        'boundary': base_lambdas['boundary'] * adaptive_weights['boundary'],
    }
    lbfgs = torch.optim.LBFGS(model.parameters(), lr=config.lbfgs_lr, max_iter=config.lbfgs_steps, line_search_fn='strong_wolfe')
    lbfgs_fixed_collocation = sample_collocation_points(config.collocation_points, device, colloc_engine)
    lbfgs_fixed_boundary = sample_boundary_points(config.boundary_points, device, bound_engine)

    def closure() -> Tensor:
        lbfgs.zero_grad(set_to_none=True)
        predicted = model(dataset.features)
        data_loss = F.mse_loss(predicted, dataset.concentration)
        physics_loss = compute_pde_residual(lbfgs_fixed_collocation, model, config.wind_u, config.wind_v, config.diffusion)
        boundary_loss = model(lbfgs_fixed_boundary).square().mean()
        loss = final_weights['data'] * data_loss + final_weights['physics'] * physics_loss + final_weights['boundary'] * boundary_loss
        loss.backward()
        return loss

    if config.lbfgs_steps > 0:
        try:
            lbfgs.step(closure)
            ema.update(model)
        except Exception as e:
            LOGGER.warning("L-BFGS stage failed or encountered NaNs: %s", e)

    # Finally, evaluate the EMA model
    ema.copy_to(model)
    return model, history


def evaluate_source_probability_grid(
    model: nn.Module,
    sector: CitySector,
    device: torch.device,
    grid_size: int = 120,
    source_time: float = 0.0,
) -> SourceProbabilityMap:
    model.eval()
    x_axis = np.linspace(0.0, 1.0, grid_size, dtype=np.float32)
    y_axis = np.linspace(0.0, 1.0, grid_size, dtype=np.float32)
    x_grid, y_grid = np.meshgrid(x_axis, y_axis)
    t_grid = np.full_like(x_grid, fill_value=source_time, dtype=np.float32)

    x_tensor = torch.tensor(x_grid.reshape(-1, 1), dtype=torch.float32, device=device)
    y_tensor = torch.tensor(y_grid.reshape(-1, 1), dtype=torch.float32, device=device)
    t_tensor = torch.tensor(t_grid.reshape(-1, 1), dtype=torch.float32, device=device)

    with torch.no_grad():
        concentration = model(x_tensor, y_tensor, t_tensor)
        concentration_grid = concentration.detach().cpu().numpy().reshape(grid_size, grid_size)

    baseline = float(np.percentile(concentration_grid, 5.0))
    source_scores = np.clip(concentration_grid - baseline, a_min=0.0, a_max=None)
    source_scores = np.square(source_scores)
    score_sum = float(np.sum(source_scores))
    if score_sum <= 0.0 or not np.isfinite(score_sum):
        LOGGER.warning("Probability scores were degenerate; returning a uniform map.")
        probabilities = np.full_like(source_scores, 1.0 / source_scores.size)
    else:
        probabilities = source_scores / score_sum

    latitudes, longitudes = normalized_to_lat_lon(x_grid, y_grid, sector)
    return SourceProbabilityMap(
        latitudes=latitudes,
        longitudes=longitudes,
        probabilities=probabilities.astype(np.float32),
    )


def estimate_source_from_probability_map(
    probability_map: SourceProbabilityMap,
) -> tuple[float, float, float]:
    max_index = np.unravel_index(
        int(np.argmax(probability_map.probabilities)),
        probability_map.probabilities.shape,
    )
    return (
        float(probability_map.latitudes[max_index]),
        float(probability_map.longitudes[max_index]),
        float(probability_map.probabilities[max_index]),
    )


def save_probability_map(
    probability_map: SourceProbabilityMap,
    output_path: Path,
) -> None:
    np.savez_compressed(
        output_path,
        latitudes=probability_map.latitudes,
        longitudes=probability_map.longitudes,
        probabilities=probability_map.probabilities,
    )
    LOGGER.info("Saved source probability map to %s", output_path)


@dataclass(frozen=True)
class SourceUncertainty:
    """Uncertainty quantification for a PINN source estimate."""
    confidence: float          # 0-1, higher = more peaked probability surface
    stddev_meters: float       # spatial spread of probability mass around peak
    peak_probability: float    # raw max probability value
    effective_cells: float     # inverse participation ratio (1 = perfect peak)


def compute_source_uncertainty(
    probability_map: SourceProbabilityMap,
) -> SourceUncertainty:
    """Derive confidence and spatial standard deviation from a probability map.

    Confidence is based on the *inverse participation ratio* (IPR):
        IPR = 1 / sum(p_i^2)
    For a perfectly peaked map (all mass on one cell), IPR = 1.
    For a uniform map over N cells, IPR = N.
    We normalize:  confidence = 1 - (IPR - 1) / (N - 1), clipped to [0, 1].

    The spatial standard deviation is the probability-weighted RMS distance
    (in meters) from the peak coordinate.
    """
    probs = probability_map.probabilities.ravel().astype(np.float64)
    lats = probability_map.latitudes.ravel().astype(np.float64)
    lons = probability_map.longitudes.ravel().astype(np.float64)
    n_cells = len(probs)

    # --- Effective cells (IPR) ---
    sum_p2 = float(np.sum(probs ** 2))
    if sum_p2 <= 0.0 or not np.isfinite(sum_p2):
        return SourceUncertainty(
            confidence=0.0,
            stddev_meters=float("inf"),
            peak_probability=0.0,
            effective_cells=float(n_cells),
        )

    ipr = 1.0 / sum_p2
    # Normalize to [0, 1]: 1 when ipr=1 (perfect peak), 0 when ipr=N (uniform)
    confidence = max(0.0, min(1.0, 1.0 - (ipr - 1.0) / max(n_cells - 1.0, 1.0)))

    # --- Peak ---
    peak_idx = int(np.argmax(probs))
    peak_lat = lats[peak_idx]
    peak_lon = lons[peak_idx]
    peak_prob = float(probs[peak_idx])

    # --- Spatial stddev (meters) ---
    # Approximate meter offsets from peak using simple lat/lon scaling
    meters_per_deg_lat = 111_320.0
    meters_per_deg_lon = meters_per_deg_lat * math.cos(math.radians(peak_lat))
    dy = (lats - peak_lat) * meters_per_deg_lat
    dx = (lons - peak_lon) * meters_per_deg_lon
    dist_sq = dx ** 2 + dy ** 2
    weighted_var = float(np.sum(probs * dist_sq))
    stddev_m = math.sqrt(max(weighted_var, 0.0))

    LOGGER.info(
        "Source UQ: confidence=%.4f stddev=%.1f m effective_cells=%.1f peak_prob=%.8f",
        confidence, stddev_m, ipr, peak_prob,
    )
    return SourceUncertainty(
        confidence=confidence,
        stddev_meters=round(stddev_m, 1),
        peak_probability=peak_prob,
        effective_cells=round(ipr, 1),
    )


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train PlumeTrace PINN source inversion model.")
    parser.add_argument("--epochs", type=int, default=2500, help="Training epochs.")
    parser.add_argument("--collocation-points", type=int, default=4096)
    parser.add_argument("--learning-rate", type=float, default=1.0e-3)
    parser.add_argument("--lambda-data", type=float, default=1.0)
    parser.add_argument("--lambda-physics", type=float, default=0.10)
    parser.add_argument("--grid-size", type=int, default=120)
    parser.add_argument("--seed", type=int, default=2026)
    parser.add_argument("--output", type=Path, default=Path("plumetrace_source_probability_map.npz"))
    parser.add_argument("--load-checkpoint", type=Path, default=None, help="Path to a .pt checkpoint to load (skips training).")
    parser.add_argument("--save-checkpoint", type=Path, default=None, help="Path to save the trained .pt checkpoint.")
    return parser.parse_args()


def main() -> None:
    configure_logging()
    args = parse_args()

    config = TrainingConfig(
        epochs=args.epochs,
        learning_rate=args.learning_rate,
        lambda_data=args.lambda_data,
        lambda_physics=args.lambda_physics,
        collocation_points=args.collocation_points,
        random_seed=args.seed,
    )
    model_config = ModelConfig()

    set_reproducibility(config.random_seed)
    device = get_device()
    sector = CitySector()

    try:
        model = PlumeInversionPINN(model_config).to(device)

        if args.load_checkpoint and args.load_checkpoint.exists():
            LOGGER.info("Loading model checkpoint from %s (bypassing training)", args.load_checkpoint)
            state_dict = torch.load(args.load_checkpoint, map_location=device, weights_only=False)
            if "model_state_dict" in state_dict:
                model.load_state_dict(state_dict["model_state_dict"])
            else:
                model.load_state_dict(state_dict)
        else:
            dataset = generate_synthetic_sensor_data(sector, config, device)
            model, history = train_inversion_model(model, dataset, config, device)
            
            if args.save_checkpoint:
                LOGGER.info("Saving trained model to %s", args.save_checkpoint)
                args.save_checkpoint.parent.mkdir(parents=True, exist_ok=True)
                torch.save({"model_state_dict": model.state_dict()}, args.save_checkpoint)

        probability_map = evaluate_source_probability_grid(
            model,
            sector,
            device,
            grid_size=args.grid_size,
            source_time=0.0,
        )
        predicted_latitude, predicted_longitude, predicted_probability = estimate_source_from_probability_map(probability_map)
        save_probability_map(probability_map, args.output)

        LOGGER.info("Known source latitude=%.6f longitude=%.6f", sector.source_latitude, sector.source_longitude)
        LOGGER.info("Estimated source latitude=%.6f longitude=%.6f probability=%.8f", predicted_latitude, predicted_longitude, predicted_probability)
        
    except Exception as exc:
        LOGGER.exception("PINN inversion run failed: %s", exc)
        raise

if __name__ == "__main__":
    main()


In [ ]:
import json
import math
import os
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.quasirandom import SobolEngine

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'kaggle_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Import and run canonical seed_everything from pinn.utils
from pinn.utils import seed_everything
seed_everything(2026, deterministic=True)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
GPU_COUNT = torch.cuda.device_count()
print('Torch:', torch.__version__)
print('Device:', DEVICE)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', GPU_COUNT)
for i in range(GPU_COUNT):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
if GPU_COUNT < 2:
    print('Warning: Kaggle is not currently exposing two GPUs. Enable Accelerator: GPU T4 x2 for the intended run.')

try:
    import mlflow
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow'], check=True, timeout=120)
        import mlflow
    except Exception as exc:
        mlflow = None
        print(f'mlflow unavailable ({exc}); continuing without experiment tracking.')

if mlflow is not None:
    mlflow.set_tracking_uri(f"sqlite:///{OUTPUT_DIR / 'mlflow.db'}")
    mlflow.set_experiment('plumetrace_pinn')
    print('MLflow tracking DB:', OUTPUT_DIR / 'mlflow.db')

def mlflow_start_run(run_name: str) -> None:
    if mlflow is not None:
        mlflow.start_run(run_name=run_name)

def mlflow_end_run() -> None:
    if mlflow is not None:
        mlflow.end_run()

def mlflow_log_params(params: dict[str, Any]) -> None:
    if mlflow is not None:
        safe = {k: (v if isinstance(v, (int, float, str, bool)) else str(v)) for k, v in params.items()}
        mlflow.log_params(safe)

def mlflow_log_metrics(metrics: dict[str, float], step: int) -> None:
    if mlflow is not None:
        mlflow.log_metrics(metrics, step=step)

def mlflow_log_artifact(path: Path) -> None:
    if mlflow is not None and path.exists():
        mlflow.log_artifact(str(path))


## Training Configuration

`ModelConfig` controls the network architecture (default: `fourier_residual`). `TrainingConfig` controls everything about the optimization: the AdamW stage (`epochs`, `learning_rate`, `warmup_epochs`), the L-BFGS fine-tune stage (`lbfgs_steps`), RAR sampling (`rar_*`), SoftAdapt (`softadapt_*`), and EMA (`ema_decay`).

`COLLOCATION_POINTS` defaults to at least 2,000 as required by the hackathon spec. On two T4 GPUs, this notebook uses 8,192 collocation points per epoch to make better use of the runtime. Set it back to `2000` if you want an exact match to the local demo script.

In [ ]:
from pinn.config import SensorStation, CitySector, ModelConfig, TrainingConfig, SENSOR_STATIONS, SECTOR

MODEL_CONFIG = ModelConfig()
CONFIG = TrainingConfig(collocation_points=8192 if GPU_COUNT >= 2 else 2000)
print(MODEL_CONFIG)
print(CONFIG)


In [ ]:
from pinn.utils import (
    lat_lon_to_normalized as _lat_lon_to_normalized,
    normalized_to_lat_lon as _normalized_to_lat_lon,
    haversine_m,
    normalize_scores
)

def lat_lon_to_normalized(latitude, longitude, sector):
    return _lat_lon_to_normalized(latitude, longitude, sector.lat_min, sector.lat_max, sector.lon_min, sector.lon_max)

def normalized_to_lat_lon(x, y, sector):
    return _normalized_to_lat_lon(x, y, sector.lat_min, sector.lat_max, sector.lon_min, sector.lon_max)

def analytic_advection_diffusion_plume(
    x: np.ndarray,
    y: np.ndarray,
    t: np.ndarray,
    source_x: float | np.ndarray,
    source_y: float | np.ndarray,
    wind_u: float,
    wind_v: float,
    diffusion: float,
    source_strength: float = 1.0,
    initial_spread: float = 0.035,
) -> np.ndarray:
    effective_time = np.maximum(t, 0.0) + initial_spread
    advected_x = source_x + wind_u * t
    advected_y = source_y + wind_v * t
    radial_distance_squared = (x - advected_x) ** 2 + (y - advected_y) ** 2
    denominator = 4.0 * math.pi * diffusion * effective_time
    exponent = -radial_distance_squared / (4.0 * diffusion * effective_time)
    return source_strength * np.exp(exponent) / denominator

@dataclass
class SyntheticSensorDataset:
    features: Tensor
    concentration: Tensor
    rows: list[dict[str, Any]]
    concentration_scale_ppb: float
    source_x: float
    source_y: float

def generate_synthetic_sensor_data(sector: CitySector, config: TrainingConfig, device: torch.device) -> SyntheticSensorDataset:
    source_x_arr, source_y_arr = lat_lon_to_normalized(sector.source_latitude, sector.source_longitude, sector)
    source_x = float(np.asarray(source_x_arr))
    source_y = float(np.asarray(source_y_arr))
    rows: list[dict[str, Any]] = []
    x_values: list[float] = []
    y_values: list[float] = []
    t_values: list[float] = []
    plume_values: list[float] = []
    sample_times = np.linspace(0.05, 1.0, config.sensor_time_samples, dtype=np.float32)
    for sensor in SENSOR_STATIONS:
        sensor_x, sensor_y = lat_lon_to_normalized(sensor.latitude, sensor.longitude, sector)
        sx = float(np.asarray(sensor_x))
        sy = float(np.asarray(sensor_y))
        for elapsed_time in sample_times:
            signal = analytic_advection_diffusion_plume(
                np.asarray(sx, dtype=np.float32),
                np.asarray(sy, dtype=np.float32),
                np.asarray(float(elapsed_time), dtype=np.float32),
                source_x,
                source_y,
                config.wind_u,
                config.wind_v,
                config.diffusion,
            )
            x_values.append(sx)
            y_values.append(sy)
            t_values.append(float(elapsed_time))
            plume_values.append(float(signal))
    plume = np.asarray(plume_values, dtype=np.float32)
    normalized_signal = plume / max(float(plume.max()), 1.0e-8)
    background_so2_ppb = 7.5
    event_scale_ppb = 180.0
    noise = np.random.normal(loc=0.0, scale=1.65, size=normalized_signal.shape)
    so2_ppb = np.clip(background_so2_ppb + event_scale_ppb * normalized_signal + noise, 0.0, None)
    concentration_scale_ppb = float(np.max(so2_ppb))
    target = (so2_ppb / concentration_scale_ppb).astype(np.float32)
    for i, (x, y, t, so2, c) in enumerate(zip(x_values, y_values, t_values, so2_ppb, target)):
        sensor = SENSOR_STATIONS[i // config.sensor_time_samples]
        rows.append({
            'sensor_id': sensor.sensor_id,
            'latitude': sensor.latitude,
            'longitude': sensor.longitude,
            'x': float(x),
            'y': float(y),
            'elapsed_time': float(t),
            'so2_ppb': float(so2),
            'normalized_concentration': float(c),
        })
    features = torch.tensor(np.column_stack([x_values, y_values, t_values]), dtype=torch.float32, device=device)
    concentration = torch.tensor(target, dtype=torch.float32, device=device).view(-1, 1)
    print(f'Generated {len(rows)} synthetic sensor readings from {len(SENSOR_STATIONS)} sensors')
    print(f'Known source lat/lon: ({sector.source_latitude:.6f}, {sector.source_longitude:.6f})')
    return SyntheticSensorDataset(features, concentration, rows, concentration_scale_ppb, source_x, source_y)

dataset = generate_synthetic_sensor_data(SECTOR, CONFIG, DEVICE)
pd.DataFrame(dataset.rows).head()


In [ ]:
from pinn_engine import PlumeInversionPINN, compute_pde_residual, sample_collocation_points, sample_boundary_points, rar_select_points

base_model = PlumeInversionPINN(MODEL_CONFIG).to(DEVICE)
if GPU_COUNT >= 2:
    model = nn.DataParallel(base_model)
    print('Using torch.nn.DataParallel across', GPU_COUNT, 'GPUs')
else:
    model = base_model
    print('Using single-device model')

sum_params = sum(p.numel() for p in model.parameters())
print(f'Architecture: {MODEL_CONFIG.arch_type} | Trainable parameters: {sum_params}')


In [ ]:
from pinn_engine import train_inversion_model

mlflow_start_run(run_name=f'plumetrace_{MODEL_CONFIG.arch_type}_{int(time.time())}')
mlflow_log_params({**asdict(CONFIG), **{f'model_{k}': v for k, v in asdict(MODEL_CONFIG).items()}})

model, history = train_inversion_model(model, dataset, CONFIG, DEVICE)
history_df = pd.DataFrame(history)
history_df.tail()


In [ ]:
def evaluate_pinn_probability_grid(model: nn.Module, sector: CitySector, device: torch.device, grid_size: int) -> dict[str, np.ndarray]:
    model.eval()
    x_axis = np.linspace(0.0, 1.0, grid_size, dtype=np.float32)
    y_axis = np.linspace(0.0, 1.0, grid_size, dtype=np.float32)
    x_grid, y_grid = np.meshgrid(x_axis, y_axis)
    t_grid = np.zeros_like(x_grid, dtype=np.float32)
    features = torch.tensor(np.column_stack([x_grid.ravel(), y_grid.ravel(), t_grid.ravel()]), dtype=torch.float32, device=device)
    preds: list[np.ndarray] = []
    with torch.no_grad():
        for chunk in torch.split(features, 32768):
            preds.append(model(chunk).detach().cpu().numpy())
    field = np.concatenate(preds, axis=0).reshape(grid_size, grid_size)
    probabilities = normalize_scores(field)
    latitudes, longitudes = normalized_to_lat_lon(x_grid, y_grid, sector)
    return {'latitudes': latitudes, 'longitudes': longitudes, 'probabilities': probabilities, 'field': field}

def evaluate_physics_fused_posterior(pinn_map: dict[str, np.ndarray], dataset: SyntheticSensorDataset, config: TrainingConfig, temperature: float = 0.010) -> dict[str, np.ndarray]:
    rows = pd.DataFrame(dataset.rows)
    obs_x = rows['x'].to_numpy(dtype=np.float32)[None, :]
    obs_y = rows['y'].to_numpy(dtype=np.float32)[None, :]
    obs_t = rows['elapsed_time'].to_numpy(dtype=np.float32)[None, :]
    obs_c = rows['normalized_concentration'].to_numpy(dtype=np.float32)[None, :]
    source_x = ((pinn_map['longitudes'] - SECTOR.lon_min) / (SECTOR.lon_max - SECTOR.lon_min)).reshape(-1, 1).astype(np.float32)
    source_y = ((pinn_map['latitudes'] - SECTOR.lat_min) / (SECTOR.lat_max - SECTOR.lat_min)).reshape(-1, 1).astype(np.float32)
    signal = analytic_advection_diffusion_plume(obs_x, obs_y, obs_t, source_x, source_y, config.wind_u, config.wind_v, config.diffusion)
    signal = signal / np.maximum(signal.max(axis=1, keepdims=True), 1.0e-8)
    mse = np.mean((signal - obs_c) ** 2, axis=1).reshape(pinn_map['probabilities'].shape)
    likelihood = np.exp(-mse / max(temperature, 1.0e-6))
    neural_prior = np.power(normalize_scores(pinn_map['probabilities']) + 1.0e-12, 0.35)
    fused = normalize_scores(neural_prior * likelihood)
    return {**pinn_map, 'probabilities': fused, 'candidate_mse': mse, 'physics_likelihood': likelihood}

def estimate_peak(probability_map: dict[str, np.ndarray]) -> dict[str, float]:
    p = probability_map['probabilities']
    idx = np.unravel_index(int(np.argmax(p)), p.shape)
    lat = float(probability_map['latitudes'][idx])
    lon = float(probability_map['longitudes'][idx])
    distance = haversine_m(SECTOR.source_latitude, SECTOR.source_longitude, lat, lon)
    return {'latitude': lat, 'longitude': lon, 'probability': float(p[idx]), 'distance_meters': distance}

# Evaluate with the trained model (which already holds EMA weights)
model.eval()
pinn_map = evaluate_pinn_probability_grid(model, SECTOR, DEVICE, CONFIG.grid_size)
fused_map = evaluate_physics_fused_posterior(pinn_map, dataset, CONFIG)
pinn_peak = estimate_peak(pinn_map)
fused_peak = estimate_peak(fused_map)
print('Raw PINN peak (EMA weights):', pinn_peak)
print('Fused physics+PINN peak:', fused_peak)


In [ ]:
checkpoint_path = OUTPUT_DIR / 'plumetrace_pinn_checkpoint.pt'
probability_map_path = OUTPUT_DIR / 'plumetrace_source_probability_map.npz'
history_path = OUTPUT_DIR / 'plumetrace_training_history.csv'
summary_path = OUTPUT_DIR / 'plumetrace_validation_summary.json'
figure_path = OUTPUT_DIR / 'plumetrace_source_probability.png'
loss_curve_path = OUTPUT_DIR / 'plumetrace_training_curves.png'

# The checkpoint stores the EMA-smoothed weights.
# SECURITY NOTE: torch.save here uses weights_only=False because it serializes non-tensor configuration dicts.
# This is acceptable for locally-produced checkpoints but poses execution risks if loading untrusted files.
torch.save(
    {
        'model_state_dict': unwrap_model(model).state_dict(),
        'model_config': asdict(MODEL_CONFIG),
        'training_config': asdict(CONFIG),
        'city_sector': asdict(SECTOR),
        'sensor_stations': [asdict(s) for s in SENSOR_STATIONS],
        'pinn_peak': pinn_peak,
        'fused_peak': fused_peak,
        'torch_version': torch.__version__,
        'gpu_count': GPU_COUNT,
        'gpu_names': [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)] if GPU_COUNT > 0 else [],
    },
    checkpoint_path,
)
np.savez_compressed(
    probability_map_path,
    latitudes=fused_map['latitudes'],
    longitudes=fused_map['longitudes'],
    probabilities=fused_map['probabilities'],
    raw_pinn_probabilities=pinn_map['probabilities'],
    candidate_mse=fused_map['candidate_mse'],
)
history_df.to_csv(history_path, index=False)

adamw_history = [h for h in history if not math.isnan(h['data_loss'])]
validation_summary = {
    'known_source': {'latitude': SECTOR.source_latitude, 'longitude': SECTOR.source_longitude},
    'architecture': MODEL_CONFIG.arch_type,
    'raw_pinn_peak': pinn_peak,
    'fused_physics_pinn_peak': fused_peak,
    'final_losses': history[-1],
    'loss_reduction_ratio': adamw_history[-1]['total_loss'] / adamw_history[0]['total_loss'],
    'lbfgs_final_total_loss': history[-1]['total_loss'],
    'probability_sum': float(fused_map['probabilities'].sum()),
    'all_probabilities_finite': bool(np.isfinite(fused_map['probabilities']).all()),
    'pass_distance_threshold_250m': bool(fused_peak['distance_meters'] <= 250.0),
}
summary_path.write_text(json.dumps(validation_summary, indent=2), encoding='utf-8')

plt.figure(figsize=(9, 7))
plt.contourf(fused_map['longitudes'], fused_map['latitudes'], fused_map['probabilities'], levels=32, cmap='inferno')
plt.colorbar(label='Source probability')
plt.scatter([SECTOR.source_longitude], [SECTOR.source_latitude], c='cyan', s=90, marker='*', label='Known source')
plt.scatter([fused_peak['longitude']], [fused_peak['latitude']], c='white', s=70, marker='x', label='Predicted source')
for station in SENSOR_STATIONS:
    plt.scatter([station.longitude], [station.latitude], c='lime', s=35)
    plt.text(station.longitude + 0.00015, station.latitude + 0.00015, station.sensor_id, color='white', fontsize=8)
plt.title(f'PlumeTrace fused physics + PINN source probability ({MODEL_CONFIG.arch_type})')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(figure_path, dpi=180)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
epochs_axis = history_df['epoch']

ax = axes[0, 0]
ax.semilogy(epochs_axis, history_df['total_loss'], label='total', linewidth=1.5)
ax.semilogy(epochs_axis, history_df['data_loss'], label='data', alpha=0.8)
ax.semilogy(epochs_axis, history_df['physics_loss'], label='physics', alpha=0.8)
ax.semilogy(epochs_axis, history_df['boundary_loss'], label='boundary', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)'); ax.set_title('Loss components'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.plot(epochs_axis, history_df['weight_data'], label='w_data', alpha=0.8)
ax.plot(epochs_axis, history_df['weight_physics'], label='w_physics', alpha=0.8)
ax.plot(epochs_axis, history_df['weight_boundary'], label='w_boundary', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('SoftAdapt weight'); ax.set_title('Adaptive loss weights'); ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(epochs_axis, history_df['learning_rate'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning rate'); ax.set_title('AdamW warmup + cosine schedule')

ax = axes[1, 1]
ax.plot(epochs_axis, history_df['grad_norm'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Gradient norm (pre-clip)'); ax.set_title('Gradient norm')

fig.suptitle(f'PlumeTrace training diagnostics ({MODEL_CONFIG.arch_type})')
fig.tight_layout()
fig.savefig(loss_curve_path, dpi=160)
plt.show()

print(json.dumps(validation_summary, indent=2))
for artifact in (checkpoint_path, probability_map_path, history_path, summary_path, figure_path, loss_curve_path):
    mlflow_log_artifact(artifact)
mlflow_end_run()


In [ ]:
# SECURITY NOTE: torch.load here uses weights_only=False because the checkpoint contains non-tensor configuration dataclasses.
# We explicitly verify that the file is locally generated and trusted.
state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
reloaded_model_config = ModelConfig(**state['model_config'])
reloaded = PlumeInversionPINN(reloaded_model_config).to(DEVICE)
reloaded.load_state_dict(state['model_state_dict'])
reloaded.eval()
test_feature = torch.tensor([[dataset.source_x, dataset.source_y, 0.0]], dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    source_concentration = float(reloaded(test_feature).cpu().item())
print('Reloaded checkpoint OK. Architecture:', reloaded_model_config.arch_type)
print('Predicted normalized C at known source, t=0:', source_concentration)


In [ ]:
# --- DataParallel Double-Backward Verification Cell ---
if torch.cuda.device_count() >= 2:
    print('Verifying DataParallel double-backward equivalence...')
    
    # 1. Run forced 1-GPU
    seed_everything(2026, deterministic=True)
    model_single = PlumeInversionPINN(MODEL_CONFIG).to(DEVICE)
    dataset_single = generate_synthetic_sensor_data(SECTOR, CONFIG, DEVICE)
    _, history_single = train_inversion_model(model_single, dataset_single, CONFIG, DEVICE, epochs_override=20)
    
    # 2. Run forced 2-GPU with nn.DataParallel
    seed_everything(2026, deterministic=True)
    model_dp = PlumeInversionPINN(MODEL_CONFIG).to(DEVICE)
    model_dp_wrapped = nn.DataParallel(model_dp)
    dataset_dp = generate_synthetic_sensor_data(SECTOR, CONFIG, DEVICE)
    _, history_dp = train_inversion_model(model_dp_wrapped, dataset_dp, CONFIG, DEVICE, epochs_override=20)
    
    # 3. Compare trajectories
    single_losses = [h['total_loss'] for h in history_single[:20]]
    dp_losses = [h['total_loss'] for h in history_dp[:20]]
    
    mismatch = False
    for epoch, (l_single, l_dp) in enumerate(zip(single_losses, dp_losses), 1):
        diff = abs(l_single - l_dp)
        print(f'Epoch {epoch:02d} | Single-GPU Loss: {l_single:.6f} | DataParallel Loss: {l_dp:.6f} | Diff: {diff:.6f}')
        if diff > 1e-4:
            mismatch = True
            
    if mismatch:
        print('FAIL: DataParallel double-backward discrepancy detected!')
    else:
        print('PASS: DataParallel double-backward trajectories match within tolerance.')
else:
    print('Skipping DataParallel double-backward check: Less than 2 GPUs available.')
